# Can a sub-1 MB student carry a narrow task at the teacher's own resolution?

**Runtime → Change runtime type → A100.**

~2 h for the two cells that matter; ~5 h for the full ladder. Checkpoints and
results go to Drive after every configuration, so a disconnect costs one config.

## What this settles

`TINY_FINDINGS.md` §3 trained students against the teacher's **K+1 posteriors over
the user's labels** — not its embedding, which is what both closed distillation
attempts matched and what `PRUNE_FINDINGS.md` says does not predict downstream
accuracy. Every student failed. But every student also ran at **128–224 px against a
teacher at 518**, so the result conflates two different claims:

| claim | what it would mean |
|---|---|
| capacity | 138k parameters cannot carry a 14-way congener task |
| acuity | the student could not resolve what it needed to see |

The second is demonstrably live. One resolution step — same model, same data —
moved the crowded arm from **exactly 0.000 to 0.174**, and moved the 0.14 MB
student's top-1 by **+7.9pp**. At 518 px the student sees precisely what the teacher
sees, and any remaining gap is capacity, architecture and training.

## The declared verdict

From `TINY_PREREG.md`, fixed before any student was trained and carried in the
bundle rather than typed here, so this notebook cannot drift from it:

| arm | teacher `label_share` | student verdict |
|---|---|---|
| separated · 14 genera | 0.8982 | **passes** at ≥ 0.8482 (within 5pp) |
| crowded · 8 *Sedum* + 6 *Trifolium* | 0.5537 | **fails** below 0.4037 (>15pp under) |

The amended re-close condition needs a student trained at the teacher's own input
resolution — which is the only thing this notebook adds, and the only thing MPS
could not deliver in reasonable time.

> **Report latency beside bytes.** `plantclef24` is a third of BioCLIP-2's parameters
> and roughly *twice* its latency, entirely because it runs at 518 px. A student that
> only works at 518 inherits that: it would be tiny as a file and not necessarily
> tiny as a computation, which for a constrained device is the budget that actually
> binds. §7 times every configuration so a positive result lands on two axes.


## 1. Environment and Drive


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install 'narrowcast @ git+https://github.com/semajyllek/narrowcast.git' \
                pandas pyarrow


In [ ]:
import os, json, time, tarfile, pathlib, subprocess
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from google.colab import drive
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive/tiny_student')
DRIVE.mkdir(parents=True, exist_ok=True)
DEV = 'cuda'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print('narrowcast', __import__('narrowcast').__file__)


## 2. The research repo

Only for the prereg text and the student architecture — the architecture is
imported rather than retyped so the notebook and the local pilot cannot diverge.


In [ ]:
REPO = pathlib.Path('narrowcast-plantid')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/semajyllek/narrowcast-plantid.git'], check=True)
import sys; sys.path.insert(0, str(REPO))
from analysis.distil_student import Student, build_model, size_report

for w in (0.5, 1.0, 2.0):
    n, mb32, mb8 = size_report(Student(15, w))
    print(f'width {w}: {n:,} params · {mb32:.2f} MB fp32 · {mb8:.2f} MB int8')


## 3. The bundle

Built locally by `analysis/make_distil_bundle.py` and uploaded once. It carries
every image at short side 518, the teacher's posteriors over the transfer set, and
the evaluation truth/bucket/cluster/group arrays.

**The teacher never runs here.** Its posteriors are `predict_proba` over vectors the
research repo already had cached, so the GPU is spent entirely on the student and
the teacher being compared against is bit-identical to the one measured locally.

```bash
# on your machine, once
PYTHONPATH=. .venv/bin/python -m analysis.make_distil_bundle \
    --out /tmp/tiny_bundle --tar /tmp/tiny_bundle.tar
# then upload /tmp/tiny_bundle.tar to Drive at MyDrive/tiny_student/
```


In [ ]:
TAR = DRIVE / 'tiny_bundle.tar'
LOCAL = pathlib.Path('/content/tiny_bundle')
assert TAR.exists(), f'upload the bundle to {TAR} first'
if not LOCAL.exists():
    t0 = time.time()
    with tarfile.open(TAR) as t:
        t.extractall('/content')
    print(f'extracted in {time.time()-t0:.0f}s')

MAN = pd.read_parquet(LOCAL / 'manifest.parquet')
Z = np.load(LOCAL / 'teacher.npz', allow_pickle=True)
SUMMARY = json.loads((LOCAL / 'summary.json').read_text())
IMG = LOCAL / 'images'

THR = SUMMARY['thresholds']
for arm, d in SUMMARY['arms'].items():
    print(f"{arm:10s} transfer {d['n_transfer']:6d}  eval {d['n_eval']:6d}  "
          f"teacher label_share {d['teacher']['label_share']:.4f}")
print('\nthresholds:', THR)


## 4. Data pipeline

Augmentation on the training side only. **Colour jitter is deliberately absent** —
several of these species separate on flower colour, and jittering it would teach the
student to ignore the feature the task turns on.


In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


class Frames(Dataset):
    def __init__(self, files, targets=None, train=True, px=518):
        self.files, self.targets, self.train, self.px = list(files), targets, train, px

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        with Image.open(IMG / self.files[i]) as im:
            im = im.convert('RGB')
            if self.train:
                w, h = im.size
                s = np.random.uniform(0.7, 1.0)
                nw, nh = int(w * s), int(h * s)
                x0 = np.random.randint(0, max(1, w - nw + 1))
                y0 = np.random.randint(0, max(1, h - nh + 1))
                im = im.crop((x0, y0, x0 + nw, y0 + nh))
                if np.random.rand() < 0.5:
                    im = im.transpose(Image.FLIP_LEFT_RIGHT)
            im = im.resize((self.px, self.px), Image.BILINEAR)
            x = torch.from_numpy(np.asarray(im, dtype=np.float32).copy() / 255.0)
        x = (x.permute(2, 0, 1) - MEAN) / STD
        return x if self.targets is None else (x, torch.from_numpy(self.targets[i]))


def files_for(arm, role):
    m = MAN[(MAN.arm == arm) & (MAN.role == role)].sort_values('idx')
    return m.file.to_numpy()


## 5. Scoring, through narrowcast's own path

The student's posteriors are replayed into `build.score_frame` in evaluation-row
order, so the cascade, the thresholds and the cluster bootstrap are the same code
that produced the teacher's numbers. A comparison through two different measurement
procedures would not be a comparison of two models.


In [ ]:
from narrowcast import build as B


class Precomputed:
    def __init__(self, classes, proba):
        self.classes_ = np.asarray(classes)
        self._p = np.asarray(proba, dtype=float)

    def predict_proba(self, X):
        assert len(X) == len(self._p), 'row order desynchronised'
        return self._p


def score(arm, probs):
    truth, bucket = Z[f'{arm}_truth'], Z[f'{arm}_bucket']
    ds = B.Dataset(X_train=np.zeros((1, 1)), y_train=np.array([B.OTHER]),
                   frame=pd.DataFrame(), X_eval=np.zeros((len(truth), 1)),
                   truth=truth, bucket=bucket,
                   counts={'in_catalog': int((bucket == 'in_catalog').sum())},
                   cluster=Z[f'{arm}_cluster'], group=Z[f'{arm}_group'])
    frame = B.score_frame(Precomputed(Z[f'{arm}_classes'], probs), ds)
    return B.fit_and_measure(frame, p_ood=SUMMARY['p_ood'])


## 6. Train one configuration


In [ ]:
def train(arm, px, init='scratch', width=1.0, epochs=60, bs=None, lr=3e-3,
          workers=8):
    """Distil the fitted task at one resolution. Returns the scored metrics."""
    classes = Z[f'{arm}_classes']
    soft = Z[f'{arm}_soft']
    bs = bs or (24 if px >= 518 else 48 if px >= 320 else 96)

    model = build_model(len(classes), width, init).to(DEV)
    n, mb32, mb8 = size_report(model)
    print(f'[{arm} · {px}px · {init}] {n:,} params · {mb8:.2f} MB int8 · bs {bs}',
          flush=True)

    dl = DataLoader(Frames(files_for(arm, 'transfer'), soft, True, px),
                    batch_size=bs, shuffle=True, num_workers=workers,
                    drop_last=True, pin_memory=True, persistent_workers=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr,
                                                total_steps=max(1, epochs * len(dl)))
    scaler = torch.amp.GradScaler('cuda')

    model.train()
    for ep in range(epochs):
        t0, tot, seen = time.time(), 0.0, 0
        for xb, yb in dl:
            xb, yb = xb.to(DEV, non_blocking=True), yb.to(DEV, non_blocking=True)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                # KL to the teacher's posterior. T**2 keeps the gradient scale
                # comparable across temperatures (Hinton et al.); without it,
                # raising T quietly lowers the learning rate.
                T = 2.0
                logp = F.log_softmax(model(xb) / T, dim=1)
                tgt = F.softmax(torch.log(yb.clamp_min(1e-8)) / T, dim=1)
                loss = F.kl_div(logp, tgt, reduction='batchmean') * T ** 2
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(xb); seen += len(xb)
        if ep % 5 == 0 or ep == epochs - 1:
            print(f'  epoch {ep+1}/{epochs}  kl {tot/max(seen,1):.4f}  '
                  f'{time.time()-t0:.0f}s', flush=True)

    model.eval()
    ev = DataLoader(Frames(files_for(arm, 'eval'), None, False, px),
                    batch_size=bs, num_workers=workers, pin_memory=True)
    probs = []
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        for xb in ev:
            probs.append(F.softmax(model(xb.to(DEV)), 1).float().cpu().numpy())
    m = score(arm, np.vstack(probs))
    m.update(params=n, mb_int8=round(mb8, 3), px=px, init=init, arm=arm,
             epochs=epochs, latency=latency(model, px))
    torch.save(model.state_dict(), DRIVE / f'student_{arm}_{px}_{init}.pt')
    return m


## 7. Latency, measured not assumed

The reason `plantclef24` is slow is 518 px, not parameters. If the student only
works at 518 it inherits that, and "sub-1 MB" becomes a storage claim with no
latency claim attached. Batch 1 is the number a constrained device sees.

This is an A100 figure and is **indicative, not a deployment measurement** — the
registry's own sizes carry the same caveat, and an ANE or an MCU will rank these
differently.


In [ ]:
def latency(model, px, n=60, warmup=15):
    out = {}
    for bs in (1, 32):
        x = torch.randn(bs, 3, px, px, device=DEV)
        with torch.no_grad():
            for _ in range(warmup):
                model(x)
            torch.cuda.synchronize(); t0 = time.time()
            for _ in range(n):
                model(x)
            torch.cuda.synchronize()
        out[f'ms_per_img_bs{bs}'] = round((time.time() - t0) / n / bs * 1000, 3)
    return out


## 8. The ladder

The two cells that decide the question are **scratch at 518** on each arm. 320 px is
the saturation check the amended condition allows as an alternative to reaching the
teacher's resolution; the 1.53 MB ImageNet-initialised arm is a near-budget ceiling,
included so a failure can be attributed to capacity rather than to initialisation.

Run the first two, look, then decide whether the rest is worth the hours.


In [ ]:
PLAN = [
    dict(arm='separated', px=518, init='scratch'),   # decides the pass
    dict(arm='crowded',   px=518, init='scratch'),   # decides the fail
    dict(arm='crowded',   px=320, init='scratch'),   # saturation check
    dict(arm='separated', px=320, init='scratch'),
    dict(arm='crowded',   px=518, init='imagenet'),  # near-budget ceiling
    dict(arm='separated', px=518, init='imagenet'),
]

RESULTS = DRIVE / 'results.json'
done = json.loads(RESULTS.read_text()) if RESULTS.exists() else []
seen = {(d['arm'], d['px'], d['init']) for d in done}

for cfg in PLAN[:2]:          # widen the slice once you have looked
    key = (cfg['arm'], cfg['px'], cfg['init'])
    if key in seen:
        print('skip (already done)', key); continue
    m = train(**cfg)
    t = SUMMARY['arms'][cfg['arm']]['teacher']['label_share']
    print(f"  -> label_share {m['label_share']:.4f} vs teacher {t:.4f} "
          f"({m['label_share']-t:+.4f})  |  {m['latency']}\n", flush=True)
    done.append({k: (v if not isinstance(v, np.generic) else v.item())
                 for k, v in m.items() if k != 'ci'})
    RESULTS.write_text(json.dumps(done, indent=2, default=float))


## 9. The verdict

Evaluated against the thresholds carried in the bundle, not against anything typed
in this notebook.


In [ ]:
def verdict(done):
    at_teacher_px = [d for d in done if d['px'] == 518]
    if not at_teacher_px:
        return 'INCOMPLETE — no run at the teacher resolution; condition (a) unmet'
    best = {}
    for d in at_teacher_px:
        b = best.get(d['arm'])
        if b is None or d['label_share'] > b['label_share']:
            best[d['arm']] = d
    if 'separated' not in best or 'crowded' not in best:
        return 'INCOMPLETE — need both arms at 518 px'
    sep, cro = best['separated']['label_share'], best['crowded']['label_share']
    lines = [f"separated {sep:.4f}  (passes at >= {THR['separated_pass_at_or_above']})",
             f"crowded   {cro:.4f}  (fails below {THR['crowded_fail_below']})"]
    if sep >= THR['separated_pass_at_or_above']:
        lines.append('\nA sub-1 MB student carries the separated task at the '
                     "teacher's own resolution.")
        lines.append('Report its latency beside its size before calling it small.')
    elif cro < THR['crowded_fail_below'] and sep < THR['separated_pass_at_or_above']:
        lines.append('\nBoth arms fail with the resolution handicap removed.')
        lines.append('Re-close condition (a) is satisfied: distillation closes.')
    else:
        lines.append('\nMixed. Neither the pass nor the re-close condition is met;')
        lines.append('record it as measured and do not round it either way.')
    return '\n'.join(lines)


print(verdict(done))
print()
for d in sorted(done, key=lambda d: (d['arm'], d['px'])):
    print(f"{d['arm']:10s} {d['px']:4d}px {d['init']:9s} "
          f"{d['mb_int8']:5.2f} MB  label_share {d['label_share']:.4f}  "
          f"top1 {d['closed_set_top1']:.4f}  headroom {d['headroom']:.4f}  "
          f"{d['latency']['ms_per_img_bs1']:6.2f} ms/img")


## 10. Back on your machine

```bash
cp ~/Downloads/results.json analysis/tiny_student_518.json
```

Then write it into `TINY_FINDINGS.md` §3 and settle `TINY_PREREG.md`'s amended
re-close condition one way or the other. The table to extend:

| student | top-1 | headroom | label-level share |
|---|---|---|---|
| 0.14 MB @ 128 px | 0.277 | 0.500 | 0.000 |
| 0.14 MB @ 224 px | 0.355 | 0.496 | 0.000 |
| 1.53 MB @ 128 px | 0.388 | 0.422 | 0.000 |
| 1.53 MB @ 224 px | 0.492 | 0.291 | 0.174 |
| **0.14 MB @ 518 px** | — | — | **this run** |
| teacher · 43.3 MB @ 518 px | 0.897 | 0.119 | 0.554 |

Three things to carry across, whichever way it lands:

1. **The `p > 0.800` cliff.** Below roughly 0.45 top-1 on a crowded set the cascade
   names no label at all, so a `label_share` of exactly 0.000 is the declared utility
   working rather than a broken run. Read top-1 to tell a dead student from a
   cautious one.
2. **Latency, not just bytes.** A student that needs 518 px is small on disk and not
   necessarily cheap to run. If §7 shows it slower than `mobileclip2_s0` at 5.7 MB,
   the registry gains a smaller *file* and no better *product*.
3. **This is one domain, one teacher, K=14, two label sets.** A pass is a result
   about task-conditional distillation on plants, not a general claim — and the
   transfer-set question is untouched either way: 29,499 unlabelled in-domain images
   is not what a user brings.
